# Lecture 8 Activity – Did MLB's 2023 rules raise stolen bases?

QSS 20 · Modern Statistical Computing · Fall 2026

## The question

Before the 2023 season, Major League Baseball changed three rules at once. A
pitch clock limits the time between pitches. A pitcher may step off the
rubber or throw to a base only twice per batter; a third failed try gives the
runner the next base. The bases grew from 15 to 18 inches on a side, which
shortens the distance between them.

All three changes favor a runner who tries to steal a base. If the rules
worked as designed, runners stole more after 2022. Each team's season totals
measure whether they did, and by how much.

> **Did base runners steal more often after the 2023 rule changes, and by how
> much?**

The MLB Stats API publishes each team's season totals. One request returns
all 30 teams for one season, so eleven requests cover 2015 to 2025.

### The MLB Stats API

Major League Baseball's Stats API, at `statsapi.mlb.com`, serves MLB's game,
player, and team records as JSON, with no key. This activity uses one of its
endpoints, `teams/stats`. Here is that endpoint's entry in the
[MLB-StatsAPI endpoint list](https://github.com/toddrob99/MLB-StatsAPI/wiki/Endpoints):

<img src="images/api-mlb-docs.png" width="560" alt="The entry Endpoint: teams_stats in the volunteer MLB-StatsAPI wiki. URL: https://statsapi.mlb.com/api/{ver}/teams/stats. Required parameters: season + group + stats. All parameters: ver, season, sportIds, group, gameType, stats, order, sortStat, fields, startDate, endDate. Note: use meta('statGroups') to look up valid values for group, and meta('statTypes') for valid values for stats">

This documentation is **unofficial**. PokeAPI, NYC Open Data, and FRED each
document their own API: the organization that runs the server writes the
documentation. MLB's own documentation site asks for a login, so this list
comes from the wiki of MLB-StatsAPI, a Python package that volunteers keep on
GitHub. The wiki says so itself: "This information is provided for reference only, without warranty or guarantee, and is not official documentation provided by MLB. This package and its author are not affiliated with MLB or any MLB team."

MLB has no part in the wiki, and it can change or remove an endpoint without
telling the wiki's authors. Use it to find endpoints and parameters, then check each detail
against what the server returns.

### Reminders

- The row unit you build is **one team in one season**.
- This activity needs `requests`. If `import requests` fails, run
  `conda activate qss20` and then `conda install requests` in a terminal.
- Save each response before you parse it, and pause between requests. MLB's
  [copyright notice](https://gdx.mlb.com/components/copyright.txt) permits
  only individual, non-commercial, non-bulk use, so keep the number of
  requests small.
- If the network fails, skip to Task 3: `data/raw/mlb/` holds the eleven
  responses saved on 2026-10-08.
- In the starter cells, replace each `...` with code.

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

pd.set_option('display.max_rows', 12)

# Sharp figures on high-resolution screens, from lecture 6.
%config InlineBackend.figure_format = 'retina'

RAW = Path('../../data/raw')
LIVE = Path('../../data/live/mlb')
LIVE.mkdir(parents=True, exist_ok=True)

MLB = 'https://statsapi.mlb.com/api/v1'
mlb_url = MLB + '/teams/stats'

# Check: the saved snapshot holds eleven seasons.
sorted(p.name for p in (RAW / 'mlb').glob('teams-hitting-*.json'))

## Before the tasks · The Padres' most recent game

Check that the API answers, and that you can read its answers, before you
collect eleven seasons. Three requests find the San Diego Padres' most recent
game and print their batting line from it. Run the three cells; there is
nothing to fill in.

In [ ]:
# 1. The teams endpoint lists every major-league team with its id.
res = requests.get(MLB + '/teams', params={'sportId': 1}, timeout=30)
[(t['id'], t['name']) for t in res.json()['teams'] if 'Padres' in t['name']]

In [ ]:
# 2. The schedule endpoint lists one team's games in a season.
# gameType: R is the regular season; F, D, L, W are the postseason rounds.
params = {'sportId': 1, 'teamId': 135, 'season': 2026,
          'gameType': 'R,F,D,L,W'}
res = requests.get(MLB + '/schedule', params=params, timeout=30)
games = [g for day in res.json()['dates'] for g in day['games']
         if g['status']['abstractGameState'] == 'Final']
last = games[-1]
away, home = last['teams']['away'], last['teams']['home']
(last['officialDate'], away['team']['name'], away['score'],
 home['team']['name'], home['score'])

In [ ]:
# 3. The boxscore endpoint holds each team's statistics for one game.
res = requests.get(f"{MLB}/game/{last['gamePk']}/boxscore", timeout=30)
box = res.json()
side = 'home' if box['teams']['home']['team']['id'] == 135 else 'away'
batting = box['teams'][side]['teamStats']['batting']
{k: batting[k] for k in ['runs', 'hits', 'homeRuns', 'baseOnBalls',
                         'strikeOuts', 'stolenBases']}

The API answered all three requests. Each request used a value from the one
before it: the team id `135`, then the game's id, `gamePk`. The filter on
`'Final'` keeps games that have ended, so a game in progress does not count.
The box score's batting record has a key named `stolenBases`, the statistic
this activity counts.

## Find `stolenBases` in the unofficial documentation

The box score covers one game; the tasks need each team's season totals.
Use the unofficial [MLB-StatsAPI endpoint list](https://github.com/toddrob99/MLB-StatsAPI/wiki/Endpoints) to find
the endpoint and its parameters.

1. Search the page for "stats" and open the entry
   [`teams_stats`](https://github.com/toddrob99/MLB-StatsAPI/wiki/Endpoints#endpoint-teams_stats). It gives the URL,
   `https://statsapi.mlb.com/api/{ver}/teams/stats` (`ver` is `v1`), and the
   required parameters: `season`, `group`, and `stats`.
2. The entry adds a note: look up the valid values of `group` with
   `meta('statGroups')`, and of `stats` with `meta('statTypes')`.
3. The [`meta`](https://github.com/toddrob99/MLB-StatsAPI/wiki/Endpoints#endpoint-meta) entry turns these into URLs:
   `/api/v1/statGroups` and `/api/v1/statTypes`. Its type `baseballStats`
   lists the name of every statistic.

In [ ]:
groups = requests.get(MLB + '/statGroups', timeout=30).json()
names = requests.get(MLB + '/baseballStats', timeout=30).json()
([g['displayName'] for g in groups],
 [n['name'] for n in names if 'stolen' in n['name'].lower()])

`group='hitting'` holds a team's batting statistics, `stats='season'` asks
for season totals, and `stolenBases` is the key to look for. `sportIds=1`
keeps the major leagues. Task 1 puts these together.

## Task 1 · One season, one path

Request the 2022 season. Check the status code and the content type before
you parse anything. Then find the path from the parsed response to one team's
`stolenBases`.

How many teams does the response hold, and how many bases did the first team
steal in 2022?



<details><summary>Hint</summary>

Print the keys at each level before you step down. `data.keys()` shows one key,
`'stats'`, which holds a list. Its first element is a dictionary with a key
`'splits'`: one element per team, each with `'team'` and `'stat'`.

</details>

In [ ]:
res = requests.get(mlb_url, params={'season': 2022, 'group': ...,
                                    'stats': ..., 'sportIds': 1},
                   timeout=30)
print(res.status_code, ...)

data = res.json()
splits = data[...][0][...]
len(splits), splits[0]['team']['name'], ...

*Your answer:*

## Task 2 · Eleven seasons, each saved

Write a loop over the seasons 2015 to 2025. For each season: send the request,
stop if it failed, save `res.text` as `LIVE / f'teams-hitting-{season}.json'`
with `encoding='utf-8'`, print the season and the number of teams, and pause
one second.

How many requests does the loop send? If the request for 2020 failed, what
would your loop do, and what would be on disk?



<details><summary>Hint</summary>

The lecture's paging loop has every part you need: `res.raise_for_status()`,
`Path.write_text`, `print`, and `time.sleep(1)`. Here the loop runs over a
known list, `range(2015, 2026)`, so it needs no `while`.

</details>

In [ ]:
for season in range(2015, 2026):
    params = {'season': season, 'group': 'hitting', 'stats': 'season',
              'sportIds': 1}
    res = requests.get(mlb_url, params=params, timeout=30)
    ...                                     # stop if the request failed
    path = LIVE / ...
    path.write_text(res.text, encoding='utf-8')     # save before parsing
    print(season, ...)                      # season and number of teams
    ...                                     # pause one second

*Your answer:*

## Task 3 · From saved files to a table

Build a DataFrame `teams` from the **saved snapshot** in `RAW / 'mlb'`, with
one row per team and season and five columns: `season`, `team`, `games`
(`gamesPlayed`), `sb` (`stolenBases`), and `cs` (`caughtStealing`).

Then print three checks. How many rows should `teams` have? Does every season
have 30 teams? Is `sb` a number?



<details><summary>Hint</summary>

Two loops: one over the files from `sorted((RAW / 'mlb').glob(...))`, and one
over the `splits` of each file. Build one dictionary per team and pass the
list of dictionaries to `pd.DataFrame`. `season` arrives as text (`'2022'`);
convert it with `int`.

</details>

In [ ]:
rows = []
for path in sorted((RAW / 'mlb').glob(...)):
    data = json.loads(path.read_text(encoding='utf-8'))
    for split in ...:
        rows.append({'season': int(split['season']),
                     'team': split['team']['name'],
                     'games': ...,
                     'sb': ...,
                     'cs': ...})
teams = pd.DataFrame(rows)

print('rows:', len(teams))
print('teams per season:', ...)
print(teams.dtypes)

*Your answer:*

## Task 4 · ❓ Steals per game

Total stolen bases by season can mislead: the 2020 season was cut to 60 games
per team. Build a table `league` with one row per season and the columns `sb`,
`games`, and `sb_per_game`, the stolen bases per team per game. Draw
`sb_per_game` by season and mark the 2023 rule change.

By how much did `sb_per_game` change from 2022 to 2023? Is 2020 unusual once
you divide by games?



<details><summary>Hint</summary>

Sum `sb` and `games` by season with `.groupby('season')[['sb', 'games']].sum()`,
then divide. `games` counts each game once for each team, so the ratio is
steals per team per game. `ax.axvline(2022.5)` draws a line between two seasons.

</details>

In [ ]:
league = teams.groupby('season')[['sb', 'games']].sum()
league = league.assign(sb_per_game=...)
league.round(3)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(league.index, ..., marker='o', color='#00693e')
ax.axvline(..., color='grey', ls='--', lw=1)
ax.set(xlabel='Season', ylabel='Stolen bases per team per game',
       title=...);

*Your answer:*

## Task 5 · What the numbers cannot tell you

Write two or three sentences. Which of the three new rules caused the rise in
stolen bases? What would you need to know to answer that, and what else could
have changed in 2023?

*Your answer:*